# Breast Cancer Prediction
Classify tumours as **benign** or **malignant** using the Wisconsin Diagnostic Breast Cancer dataset.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
sns.set_theme(style="whitegrid")

## 2. Load the data
In this dataset `0 = malignant` and `1 = benign`.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame
print(df.shape)
df.head()

In [ ]:
print(df.isnull().sum().sum(), "missing values")
df["target"].map({0: "malignant", 1: "benign"}).value_counts()

## 3. Exploratory data analysis

In [ ]:
df["target"].map({0: "malignant", 1: "benign"}).value_counts().plot(kind="bar", color=["#d9534f", "#5cb85c"])
plt.title("Class distribution"); plt.ylabel("Count"); plt.xticks(rotation=0); plt.show()

In [ ]:
cols = ["mean radius", "mean texture", "mean area", "mean concavity"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, c in zip(axes, cols):
    sns.histplot(data=df, x=c, hue=df["target"].map({0: "malignant", 1: "benign"}), kde=True, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.iloc[:, :10].corr(), annot=True, fmt=".1f", cmap="coolwarm")
plt.title("Correlation of the 10 mean features"); plt.show()

## 4. Train/test split

In [ ]:
X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 5. Compare models
Scaling is done inside a pipeline so no information leaks from the test set.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

rows = []
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5).mean()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "CV accuracy": cv,
        "Test accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    })
results = pd.DataFrame(rows).set_index("Model").round(4)
results.sort_values("Test accuracy", ascending=False)

## 6. Best model in detail

In [ ]:
best_name = results["Test accuracy"].idxmax()
best = models[best_name]
pred = best.predict(X_test)
print("Best model:", best_name)
print(classification_report(y_test, pred, target_names=["malignant", "benign"]))

sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["malignant", "benign"], yticklabels=["malignant", "benign"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix - {best_name}"); plt.show()

## 7. Feature importance (Random Forest)

In [ ]:
rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
imp.sort_values().plot(kind="barh", figsize=(7, 5), color="#337ab7")
plt.title("Top 10 features"); plt.show()

## 8. Conclusion
Write 2-3 lines in your own words: which model won, its accuracy and recall, and which features mattered most. In a medical setting, **recall on the malignant class** is especially important because a missed malignant case is costly.